In [5]:
!pip install -q transformers datasets peft accelerate bitsandbytes gradio trl huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 9.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # paste your Write token here

In [7]:
from datasets import load_dataset

dataset = load_dataset("ccdv/pubmed-summarization", "section", trust_remote_code=False)
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 119924
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6633
    })
    test: Dataset({
        features: ['article', 'abstract'],
        num_rows: 6658
    })
})
{'article': "a recent systematic analysis showed that in 2011 , 314 ( 296 - 331 ) million children younger than 5 years were mildly , moderately or severely stunted and 258 ( 240 - 274 ) million were mildly , moderately or severely underweight in the developing countries . \n in iran a study among 752 high school girls in sistan and baluchestan showed prevalence of 16.2% , 8.6% and 1.5% , for underweight , overweight and obesity , respectively . \n the prevalence of malnutrition among elementary school aged children in tehran varied from 6% to 16% . \n anthropometric study of elementary school students in shiraz revealed that 16% of them suffer from malnutrition and low body weight . \n

In [8]:
from datasets import DatasetDict

# Use small subset so Colab doesn't run out of memory
train_data = dataset["train"].select(range(5000))
val_data = dataset["validation"].select(range(500))

small_dataset = DatasetDict({
    "train": train_data,
    "validation": val_data
})

print(small_dataset)

DatasetDict({
    train: Dataset({
        features: ['article', 'abstract'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['article', 'abstract'],
        num_rows: 500
    })
})


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-1B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded!")

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Model loaded!


In [10]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [11]:
def format_example(example):
    return {
        "text": f"### Article:\n{example['article'][:1024]}\n\n### Summary:\n{example['abstract']}"
    }

train_formatted = small_dataset["train"].map(format_example)
val_formatted = small_dataset["validation"].map(format_example)

def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )
    result["labels"] = result["input_ids"].copy()
    return result

train_tokenized = train_formatted.map(tokenize, batched=True, remove_columns=train_formatted.column_names)
val_tokenized = val_formatted.map(tokenize, batched=True, remove_columns=val_formatted.column_names)

print("Dataset ready!")

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset ready!


In [17]:
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from trl import SFTTrainer

# Use 10K samples
train_data_10k = dataset["train"].select(range(10000))
val_data_10k = dataset["validation"].select(range(500))

def format_example(example):
    return {
        "text": f"### Article:\n{example['article'][:1024]}\n\n### Summary:\n{example['abstract']}"
    }

train_10k_formatted = train_data_10k.map(format_example)
val_10k_formatted = val_data_10k.map(format_example)

train_10k_tokenized = train_10k_formatted.map(tokenize, batched=True, remove_columns=train_10k_formatted.column_names)
val_10k_tokenized = val_10k_formatted.map(tokenize, batched=True, remove_columns=val_10k_formatted.column_names)

training_args = TrainingArguments(
    output_dir="./llama-medical-summarization-v2",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_steps=500,
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_10k_tokenized,
    eval_dataset=val_10k_tokenized,
    data_collator=data_collator,
)

trainer.train()

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass u

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
500,2.146278,2.156993,2.303832,0.545713,4096000.000000
1000,2.110599,2.148729,2.283354,0.547112,8192000.000000
1500,2.107425,2.145012,2.294041,0.547324,12288000.000000
1875,2.124192,2.144145,2.281631,0.547441,15360000.000000


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=1875, training_loss=2.1362525634765626, metrics={'train_runtime': 12528.6837, 'train_samples_per_second': 2.395, 'train_steps_per_second': 0.15, 'total_flos': 8.984218042368e+16, 'train_loss': 2.1362525634765626, 'epoch': 3.0})

In [18]:
from rouge_score import rouge_scorer
import numpy as np

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

model.eval()
samples = small_dataset["validation"].select(range(50))
scores = []

for sample in samples:
    input_text = f"### Article:\n{sample['article'][:512]}\n\n### Summary:\n"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=128, do_sample=False)

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_summary = generated.split("### Summary:")[-1].strip()

    score = scorer.score(sample['abstract'], generated_summary)
    scores.append(score['rougeL'].fmeasure)

print(f"Average ROUGE-L Score: {np.mean(scores):.4f}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eo

Average ROUGE-L Score: 0.1821


In [19]:
from huggingface_hub import HfApi

model.push_to_hub("harsha85018/llama-medical-summarizer")
tokenizer.push_to_hub("harsha85018/llama-medical-summarizer")
print("Pushed to HuggingFace!")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  18%|#7        | 1.22MB / 6.82MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpk6qyylb1/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Pushed to HuggingFace!


In [20]:
import gradio as gr

def summarize(article):
    input_text = f"### Article:\n{article[:512]}\n\n### Summary:\n"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    summary = generated.split("### Summary:")[-1].strip()
    return summary

demo = gr.Interface(
    fn=summarize,
    inputs=gr.Textbox(lines=10, placeholder="Paste a medical article or clinical note here...", label="Medical Article"),
    outputs=gr.Textbox(lines=5, label="Generated Summary"),
    title="Medical Text Summarizer",
    description="Fine-tuned LLaMA 3.2 (1B) with LoRA/QLoRA on 10K+ PubMed abstracts for medical text summarization.",
    examples=[
        ["Patients with type 2 diabetes mellitus have an increased risk of cardiovascular disease. This study examined the effect of metformin on cardiovascular outcomes in 1,200 patients over a 5-year period. Results showed a 23% reduction in major cardiovascular events compared to placebo."]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://06aabcf88778a525a9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
app_code = '''
import gradio as gr
import torch
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

model_id = "harsha85018/llama-medical-summarizer"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")

def summarize(article):
    input_text = f"### Article:\\n{article[:512]}\\n\\n### Summary:\\n"
    inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated.split("### Summary:")[-1].strip()

demo = gr.Interface(
    fn=summarize,
    inputs=gr.Textbox(lines=10, placeholder="Paste a medical article here...", label="Medical Article"),
    outputs=gr.Textbox(lines=5, label="Generated Summary"),
    title="Medical Text Summarizer",
    description="Fine-tuned LLaMA 3.2 (1B) with LoRA/QLoRA on 10K+ PubMed abstracts."
)

demo.launch()
'''

with open("/content/app.py", "w") as f:
    f.write(app_code)

print("app.py updated!")

app.py updated!


In [11]:
!gradio deploy

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ample_data/mnist_test.csv: 100% 18.3M/18.3M [00:00<?, ?B/s]


  ...ata/mnist_train_small.csv: 100% 36.5M/36.5M [00:00<?, ?B/s]

  ...ample_data/mnist_test.csv: 100% 18.3M/18.3M [00:00<?, ?B/s]


Processing Files (2 / 2)      : 100% 54.8M/54.8M [00:01<00:00, 53.4MB/s,   ???B/s  ]
New Data Upload               : |          |  0.00B /  0.00B,   ???B/s  
  ...ample_data/mnist_test.csv: 100% 18.3M/18.3M [00:00<?, ?B/s]
  ...ata/mnist_train_small.csv: 100% 36.5M/36.5M [00:00<?, ?B/s]
Space available at 
https://huggingface.co/spaces/harsha85018/medical-text-summarizer


In [13]:
from huggingface_hub import HfApi

api = HfApi()
api.upload_file(
    path_or_fileobj="/content/app.py",
    path_in_repo="app.py",
    repo_id="harsha85018/medical-text-summarizer",
    repo_type="space"
)
print("Pushed!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Pushed!
